# Collaborative filtering (cosine similarity) on recent movies

The classic recommenders, on the dataset made by `01_prep.ipynb` (4,000 movies released 2010-2023, 10,628 users,
~1M ratings), evaluated exactly like SASRec, TFRS and Qwen: the same 1,000 test users, full ranking, already-rated
movies skipped.

| model | idea |
|---|---|
| **ItemKNN** | movies are similar when the same users rated them (cosine similarity between the movies' user columns); a movie's score is its summed similarity to everything the user rated |
| **ItemKNN (last 20)** | the same, using only the user's 20 most recent movies |
| **UserKNN** | users are similar when they rated the same movies (cosine similarity between users' rows); score = what the k most similar users rated |

The neighbourhood size `k` is picked on the validation movie (each user's second-to-last rating).

In [1]:
import time

import numpy as np
import pandas as pd

from genrec import lookup
from genrec.data import Dataset, history, sample_test_users, target
from genrec.metrics import evaluate_scores, top_k
from genrec.models import baselines as B
from genrec.paths import ARTIFACTS_DIR, DATA_DIR, RESULTS_DIR

DATA = DATA_DIR / "ml-32m-recent"
K_GRID = [10, 20, 50, 100, 200, None]   # None = every neighbour
RECENT = 20

## 1. Data

In [2]:
ds = Dataset.load(DATA)
seqs = ds.encoded()                  # user -> item indices (1..n) in rating order
users = ds.users
test_users = sample_test_users(users)
row_of = {u: r for r, u in enumerate(users)}
title = dict(zip(range(1, ds.index.n_cols), ds.movies.set_index("MovieID").loc[ds.index.movie_ids, "display"]))

# users x movies 0/1 matrices of everything rated before the validation / test target
X = {split: B.interaction_matrix([history(seqs[u], split) for u in users], ds.index.n_cols) for split in ["val", "test"]}
print(f"{len(users):,} users, {ds.index.n_cols - 1:,} movies; density {X['test'].mean():.1%}")


def split_lists(us, split):
    return [history(seqs[u], split) for u in us], [target(seqs[u], split) for u in us]

10,628 users, 4,000 movies; density 2.3%


## 2. Tune the neighbourhood size on the validation movie (all users)

In [3]:
val_hist, val_tgt = split_lists(users, "val")
tuning, best_k = [], {}
for name in ["ItemKNN", f"ItemKNN (last {RECENT})", "UserKNN"]:
    t0 = time.time()
    for k in K_GRID:
        if name == "UserKNN":
            scores = B.userknn_scores(X["val"], list(range(len(users))), k)
        else:
            scores = B.itemknn_scores(val_hist, B.item_similarity(X["val"], k), recent=RECENT if "last" in name else None)
        tuning.append({"model": name, "k": "all" if k is None else k, **evaluate_scores(scores, val_hist, val_tgt)})
    best = max((r for r in tuning if r["model"] == name), key=lambda r: r["NDCG@10"])
    best_k[name] = None if best["k"] == "all" else best["k"]
    print(f"{name}: best k = {best['k']} (validation NDCG@10 {best['NDCG@10']:.3f}), {time.time() - t0:.0f}s")

pd.DataFrame(tuning).pivot(index="k", columns="model", values="NDCG@10").style.format("{:.3f}") \
    .set_caption("validation NDCG@10 by neighbourhood size")

ItemKNN: best k = 50 (validation NDCG@10 0.060), 13s


ItemKNN (last 20): best k = 50 (validation NDCG@10 0.066), 10s


UserKNN: best k = 200 (validation NDCG@10 0.063), 22s


model,ItemKNN,ItemKNN (last 20),UserKNN
k,,,
10,0.057,0.064,0.061
20,0.058,0.065,0.063
50,0.060,0.066,0.063
100,0.060,0.065,0.063
200,0.059,0.064,0.063
all,0.058,0.064,0.049


## 3. Test-set comparison

In [4]:
test_hist, test_tgt = split_lists(test_users, "test")
test_scores = {
    "ItemKNN": B.itemknn_scores(test_hist, B.item_similarity(X["test"], best_k["ItemKNN"])),
    f"ItemKNN (last {RECENT})": B.itemknn_scores(test_hist, B.item_similarity(X["test"], best_k[f"ItemKNN (last {RECENT})"]),
                                                 recent=RECENT),
    "UserKNN": B.userknn_scores(X["test"], [row_of[u] for u in test_users], best_k["UserKNN"]),
}
results = pd.DataFrame({name: evaluate_scores(s, test_hist, test_tgt) for name, s in test_scores.items()}).T
RESULTS_DIR.mkdir(exist_ok=True)
results.to_csv(RESULTS_DIR / "ml32m_recent_cf_test.csv")

# next to the other models on the same 1,000 users
others = [pd.read_csv(RESULTS_DIR / f, index_col=0) for f in ["ml32m_recent_sasrec_test.csv", "ml32m_recent_tfrs_test.csv"]
          if (RESULTS_DIR / f).exists()]
table = pd.concat([results, *others])
table.loc["Qwen2.5-0.5B (4 epochs)"] = {"hit@1": 0.023, "HR@10": 0.152, "NDCG@10": 0.076}   # qwen_colab.ipynb
print(f"test: last rating of {len(test_users)} users")
table.sort_values("NDCG@10")[["hit@1", "HR@10", "NDCG@10"]].style.format("{:.3f}")

test: last rating of 1000 users


,hit@1,HR@10,NDCG@10
random,0.000,0.001,0.000
most popular,0.018,0.088,0.046
TFRS two-tower,0.018,0.097,0.051
"TFRS sequential (GRU, last 20)",0.021,0.107,0.056
ItemKNN,0.021,0.118,0.063
UserKNN,0.028,0.112,0.064
ItemKNN (last 20),0.022,0.125,0.065
Qwen2.5-0.5B (4 epochs),0.023,0.152,0.076
SASRec-20,0.040,0.174,0.097
SASRec-200,0.042,0.182,0.100


## 4. Export the best CF model's top 10 for the API

Probability = the movie's share of the summed similarity over the movies the user hasn't rated.

In [5]:
best_cf = results["NDCG@10"].idxmax()
export = lookup.to_rows("cf-cosine", test_users,
                        lookup.normalized_top_k(test_scores[best_cf], test_scores[best_cf], test_hist),
                        decode=lambda i: ds.index.movie_ids[i - 1])
EXPORT = ARTIFACTS_DIR / "ml32m-recent" / "cf_cosine.csv"
EXPORT.parent.mkdir(parents=True, exist_ok=True)
export.to_csv(EXPORT, index=False)
print(f"exported {best_cf} ({len(export)} recommendations) -> artifacts/ml32m-recent/{EXPORT.name}")

exported ItemKNN (last 20) (10000 recommendations) -> artifacts/ml32m-recent/cf_cosine.csv


## 5. Example recommendations

In [6]:
for r, u in enumerate(test_users[:3]):
    hist, tgt = test_hist[r], test_tgt[r]
    print(f"user {u}  last rated: {'; '.join(title[m] for m in hist[-5:])}")
    print(f"   real next: {title[tgt]}")
    for name in test_scores:
        top = top_k(test_scores[name][r:r + 1], [hist])[0].tolist()
        verdict = f"HIT at rank {top.index(tgt) + 1}" if tgt in top else "miss"
        print(f"   {name:17s} ({verdict}): {'; '.join(title[m] for m in top[:4])} ...")
    print()

user 119156  last rated: The Martian (2015); Deadpool (2016); Star Wars: Episode VII - The Force Awakens (2015); Captain America: The Winter Soldier (2014); John Wick (2014)
   real next: Spotlight (2015)
   ItemKNN           (miss): Guardians of the Galaxy (2014); Skyfall (2012); X-Men: First Class (2011); The Hunger Games (2012) ...
   ItemKNN (last 20) (miss): Mission: Impossible - Rogue Nation (2015); Skyfall (2012); Avengers: Age of Ultron (2015); Iron Man 3 (2013) ...
   UserKNN           (miss): X-Men: First Class (2011); Guardians of the Galaxy (2014); The Hunger Games (2012); Django Unchained (2012) ...

user 130513  last rated: Safety Not Guaranteed (2012); Inception (2010); The Imitation Game (2014); Shutter Island (2010); Room (2015)
   real next: Les Misérables (2012)
   ItemKNN           (miss): The Kings of Summer (2013); The Way, Way Back (2013); Me and Earl and the Dying Girl (2015); Your Sister's Sister (2011) ...
   ItemKNN (last 20) (miss): The Kings of Summer (2013